# Day 009 | Error Handling — try / except / finally

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals | **Author:** Ripon Chandra Malo | **Date:** February 24, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `try` / `except` | Catch errors without crashing | Handle "NR" in N-value column |
| Specific exceptions | Catch the RIGHT error type | `ValueError` vs `ZeroDivisionError` |
| `else` | Runs only when NO error | Print success message |
| `finally` | ALWAYS runs (cleanup) | Save log even if processing crashes |
| Multiple `except` | Handle each error differently | Bad value vs missing file |
| `raise` | Create your own errors | Reject N60 > 100 as invalid |
| Skip-and-continue | Process good rows, log bad ones | Real data pipeline |

---

## Our Geotechnical Problem

Real-world geotechnical data is **messy**:

```csv
depth,N60,soil
1.5,5,Fill          ← good
3.0,NR,Sand         ← "NR" = No Recovery (not a number!)
4.5,12,Sand         ← good
6.0m,18,Clay        ← has extra "m" in depth
7.5,,Gravel         ← blank N-value!
9.0,35,Gravel       ← good
```

Without error handling: `int("NR")` → **CRASH**. You lose all results.

With error handling: skip the bad row, log the problem, keep processing.

### The Field Safety Analogy

```
Without error handling:              With error handling:
  One torn sample bag               One torn sample bag
  → STOP entire project             → LOG the problem
  → Lose all work                   → Use backup plan
  → Go home                         → CONTINUE drilling
```

---

## Part 1: Common Errors — What Goes Wrong

Before we learn to catch errors, let's SEE the errors Python throws. These are all real situations in geotechnical data:

| Error Type | Cause | Geotech Example |
|-----------|-------|------------------|
| `ValueError` | Can't convert type | `int("NR")` |
| `ZeroDivisionError` | Dividing by zero | `150 / N60` when N60=0 |
| `FileNotFoundError` | File doesn't exist | Wrong CSV path |
| `KeyError` | Dict key missing | `row["N60"]` but column is named `"n_value"` |
| `IndexError` | List index too big | `depths[10]` when only 5 depths |

In [ ]:
# ─────────────────────────────────────────────────────
# LET'S SEE EACH ERROR (safely, using try/except preview)
# ─────────────────────────────────────────────────────

# ERROR 1: ValueError — bad conversion
print("Error 1: ValueError")
try:
    n = int("NR")  # Driller wrote "NR" instead of a number
except ValueError as e:
    print(f"  {type(e).__name__}: {e}")
    print("  Cause: 'NR' cannot become an integer\n")

# ERROR 2: ZeroDivisionError
print("Error 2: ZeroDivisionError")
try:
    result = 150 / 0  # N60=0
except ZeroDivisionError as e:
    print(f"  {type(e).__name__}: {e}")
    print("  Cause: N60=0 at refusal depth\n")

# ERROR 3: FileNotFoundError
print("Error 3: FileNotFoundError")
try:
    with open("missing_file.csv") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"  {type(e).__name__}: {e}")
    print("  Cause: file path is wrong or file was moved\n")

# ERROR 4: KeyError — wrong dictionary key
print("Error 4: KeyError")
try:
    layer = {"depth": 3.0, "N60": 12}
    soil = layer["soil_type"]  # doesn't exist
except KeyError as e:
    print(f"  {type(e).__name__}: {e}")
    print("  Cause: column name doesn't match expected\n")

---

## Part 2: try / except — The Basic Pattern

The simplest error handling: **try** something risky, **except** catch the error.

```python
try:
    # Code that MIGHT fail
    n = int(raw_value)
except ValueError:
    # What to do if it DOES fail
    n = 0  # use a safe default
```

Think of it as a safety net under a tightrope walker:
- `try:` → the tightrope walk (might fall)
- `except:` → the safety net (catches the fall)

In [ ]:
# ─────────────────────────────────────────────────────
# BASIC try/except — Safe N-value conversion
# ─────────────────────────────────────────────────────

def safe_int(value, default=0):
    """Convert to int safely. Return default if it fails."""
    try:
        return int(value)
    except ValueError:
        return default

# Test with good and bad field data
test_values = ["22", "8", "NR", "", "15", "-", "35"]

print("Converting field N-values safely:")
for val in test_values:
    result = safe_int(val)
    flag = "✓" if result > 0 else "→ default"
    print(f"  '{val:3s}' → {result:3d}  {flag}")

print()
print("Without try/except: int('NR') would CRASH the script.")
print("With try/except:    it returns 0 and keeps going.")

---

## Part 3: Catching SPECIFIC Errors

Always catch **specific** error types. Different errors need different responses:

```python
try:
    result = compute(data)
except ValueError:
    print("Bad data format")    # handle format error
except ZeroDivisionError:
    print("Division by zero")   # handle math error
except FileNotFoundError:
    print("File missing")       # handle missing file
```

**Never** use bare `except:` (without specifying the type) — it catches everything, including bugs you WANT to see.

In [ ]:
# ─────────────────────────────────────────────────────
# MULTIPLE except — Each error handled differently
# ─────────────────────────────────────────────────────

def safe_layer_analysis(depth_str, n_str, soil):
    """
    Process one layer with specific error handling.
    Returns dict with 'status' indicating success or error type.
    """
    try:
        depth = float(depth_str)
        N60 = int(n_str)
        Vs = 97 * (N60 ** 0.314)
        ratio = 150 / N60           # might be zero!
        
        return {"depth": depth, "N60": N60, "Vs": round(Vs, 1),
                "soil": soil, "status": "OK"}
    
    except ValueError:
        return {"depth": depth_str, "N60": n_str, "soil": soil,
                "status": "BAD_VALUE",
                "note": f"Cannot convert '{n_str}' to number"}
    
    except ZeroDivisionError:
        return {"depth": float(depth_str), "N60": 0, "soil": soil,
                "Vs": 0, "status": "ZERO_N",
                "note": "N=0, division failed"}

# Test with various messy data
test_data = [
    ("1.5", "5",  "Fill"),       # OK
    ("3.0", "NR", "Sand"),       # ValueError
    ("4.5", "0",  "Clay"),       # ZeroDivisionError
    ("6.0", "18", "Sand"),       # OK
    ("7.5", "",   "Gravel"),     # ValueError (empty)
    ("9.0", "35", "Gravel"),     # OK
]

print(f"{'Depth':>6} {'N':>4} {'Soil':10s} {'Status':12s} Note")
print("─" * 50)
for d, n, s in test_data:
    result = safe_layer_analysis(d, n, s)
    note = result.get("note", "")
    print(f"{str(d):>6} {str(n):>4} {s:10s} {result['status']:12s} {note}")

---

## Part 4: `else` and `finally`

The full try/except structure has 4 parts:

```python
try:
    risky_code()           # might fail
except SomeError:
    handle_error()         # runs IF error
else:
    success_code()         # runs if NO error
finally:
    cleanup()              # ALWAYS runs — no matter what
```

| Block | When it runs | Use for |
|-------|-------------|--------|
| `try` | Always attempted | Risky operations |
| `except` | Only on error | Error recovery |
| `else` | Only on success (no error) | Success actions |
| `finally` | **ALWAYS** (error or not) | Cleanup: close files, save logs |

In [ ]:
# ─────────────────────────────────────────────────────
# try / except / else / finally — Full structure
# ─────────────────────────────────────────────────────
import csv, os
os.makedirs("geotech_data", exist_ok=True)

# Create a test file
with open("geotech_data/test_bh.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["depth", "N60", "soil"])
    writer.writerow([1.5, 5, "Fill"])
    writer.writerow([3.0, 12, "Sand"])

def load_and_process(filepath):
    """Demonstrates all four blocks."""
    print(f"  Loading: {filepath}")
    try:
        f = open(filepath, "r")
        data = list(csv.DictReader(f))
        f.close()
    except FileNotFoundError:
        print("  except: File not found!")
        data = None
    else:
        # Only runs if NO error
        print(f"  else: Success! Loaded {len(data)} rows")
    finally:
        # ALWAYS runs
        print("  finally: Cleanup done (always runs)")
    return data

# Test with existing file
print("Test 1: File exists")
result = load_and_process("geotech_data/test_bh.csv")
print()

# Test with missing file
print("Test 2: File missing")
result = load_and_process("geotech_data/nope.csv")
print()

print("Notice: 'finally' ran in BOTH cases!")

---

## Part 5: `raise` — Creating Your Own Errors

Sometimes you want to **create** an error when data doesn't make engineering sense — even though Python wouldn't crash on its own.

```python
if N60 < 0:
    raise ValueError("N60 cannot be negative!")

if N60 > 100:
    raise ValueError("N60 too high — check your data")
```

Think of it as **quality control**: Python won't complain about `N60 = -5` (it's a valid integer), but an engineer knows negative blow counts are impossible.

In [ ]:
# ─────────────────────────────────────────────────────
# RAISE — Engineering data validation
# ─────────────────────────────────────────────────────

def validate_spt(N60, depth):
    """Raise error if SPT data doesn't make engineering sense."""
    if N60 < 0:
        raise ValueError(f"N60 cannot be negative (got {N60})")
    if N60 > 100:
        raise ValueError(f"N60={N60} unreasonably high (max ~100)")
    if depth < 0:
        raise ValueError(f"Depth cannot be negative (got {depth})")
    if depth > 100:
        raise ValueError(f"Depth={depth}m too deep for SPT")
    return True

# Test validation
tests = [(22, 6.0), (-5, 3.0), (150, 6.0), (22, -2.0), (30, 9.0)]

for n, d in tests:
    try:
        validate_spt(n, d)
        print(f"  ✓ N60={n:4d}, depth={d:5.1f}m — Valid")
    except ValueError as e:
        print(f"  ✗ N60={n:4d}, depth={d:5.1f}m — {e}")

print()
print("'raise' = engineering quality control inside your code.")

---

## Part 6: The Most Important Pattern — Skip Bad, Keep Good

This is THE pattern you'll use most often in real data processing:

```python
good = []
errors = []

for row in raw_data:
    try:
        result = process(row)   # might fail
        good.append(result)     # save if good
    except (ValueError, KeyError) as e:
        errors.append(str(e))   # log if bad

print(f"{len(good)} good, {len(errors)} errors")
```

**Key insight:** The `try/except` is **inside** the loop. Each row gets its own chance. One bad row doesn't kill the rest.

In [ ]:
# ─────────────────────────────────────────────────────
# SKIP BAD ROWS, KEEP GOOD ONES — The real-world pattern
# ─────────────────────────────────────────────────────

def process_borehole_safe(raw_rows, water_table=3.2):
    """
    Process borehole data, skipping bad rows gracefully.
    Returns: (good_results, error_log)
    """
    good = []
    errors = []
    
    for i, row in enumerate(raw_rows, start=1):
        try:
            depth = float(row["depth"])
            N60 = int(row["N60"])
            
            if N60 < 0:
                raise ValueError(f"Negative N60: {N60}")
            
            Vs = 97 * (N60 ** 0.314) if N60 > 0 else 0
            
            if N60 < 4: cons = "Very Loose"
            elif N60 < 10: cons = "Loose"
            elif N60 < 30: cons = "Medium Dense"
            elif N60 < 50: cons = "Dense"
            else: cons = "Very Dense"
            
            good.append({
                "depth": depth, "N60": N60,
                "Vs": round(Vs, 1),
                "soil": row.get("soil", "Unknown"),
                "class": cons,
            })
        
        except (ValueError, KeyError) as e:
            errors.append({"row": i, "error": str(e), "data": row})
    
    return good, errors

# Simulate messy data (some rows are bad)
messy = [
    {"depth": "1.5", "N60": "5",  "soil": "Fill"},
    {"depth": "3.0", "N60": "NR", "soil": "Sand"},     # bad!
    {"depth": "4.5", "N60": "12", "soil": "Sand"},
    {"depth": "6.0", "N60": "18", "soil": "Clayey Sand"},
    {"depth": "bad", "N60": "28", "soil": "Sand"},     # bad!
    {"depth": "9.0", "N60": "35", "soil": "Gravel"},
    {"depth": "10.5","N60": "",   "soil": "Rock"},     # bad!
    {"depth": "12.0","N60": "60", "soil": "Bedrock"},
]

good, errors = process_borehole_safe(messy)

print(f"Results: {len(good)} good, {len(errors)} errors out of {len(messy)} rows\n")

print("✓ Good rows:")
print(f"  {'Dep':>5} {'N60':>4} {'Soil':14s} {'Vs':>6} Class")
print("  " + "─" * 42)
for r in good:
    print(f"  {r['depth']:4.1f}m {r['N60']:4d} {r['soil']:14s} {r['Vs']:5.1f} {r['class']}")

print(f"\n✗ Error log:")
for e in errors:
    print(f"  Row {e['row']}: {e['error']}")

---

## Part 7: Complete Application — Crash-Proof CSV Pipeline

Putting it all together: read a messy CSV, process safely, save clean results + error log.

```
messy_field_data.csv → [try/except each row] → clean_results.csv
                                              → error_log.txt
```

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 1: Create a messy CSV (simulates driller data)
# ══════════════════════════════════════════════════════

with open("geotech_data/messy_field.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["borehole", "depth_m", "N_raw", "soil"])
    writer.writerow(["BH-01", "1.5",  "5",  "Fill"])
    writer.writerow(["BH-01", "3.0",  "NR", "Sand"])        # No Recovery
    writer.writerow(["BH-01", "4.5",  "12", "Sand"])
    writer.writerow(["BH-01", "6.0m", "18", "Clayey Sand"]) # Extra 'm'
    writer.writerow(["BH-01", "7.5",  "28", "Sand"])
    writer.writerow(["BH-01", "9.0",  "",   "Gravel"])      # Empty N
    writer.writerow(["BH-01", "10.5", "50", "Rock"])
    writer.writerow(["BH-01", "12.0", "60", "Bedrock"])

print("Step 1: ✓ Created messy_field.csv")
with open("geotech_data/messy_field.csv", "r") as f:
    print(f.read())

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 2: Read the file (with error handling)
# ══════════════════════════════════════════════════════

try:
    with open("geotech_data/messy_field.csv", "r") as f:
        reader = csv.DictReader(f)
        raw_data = list(reader)
    print(f"Step 2: ✓ Loaded {len(raw_data)} rows")
except FileNotFoundError:
    print("Step 2: ✗ File not found!")
    raw_data = []

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 3: Process each row safely
# ══════════════════════════════════════════════════════

wt = 3.2
clean = []
error_log = []

for i, row in enumerate(raw_data, start=1):
    try:
        # Clean depth (handle '6.0m' → 6.0)
        depth_clean = row["depth_m"].replace("m", "").strip()
        depth = float(depth_clean)
        
        # Convert N-value (handle 'NR', empty, etc.)
        n_raw = row["N_raw"].strip()
        if n_raw == "" or n_raw.upper() in ["NR", "REF", "-"]:
            raise ValueError(f"Non-numeric N: '{row['N_raw']}'")
        
        N60 = int(n_raw)
        Vs = 97 * (N60 ** 0.314) if N60 > 0 else 0
        
        if N60 < 4: cons = "Very Loose"
        elif N60 < 10: cons = "Loose"
        elif N60 < 30: cons = "Medium Dense"
        elif N60 < 50: cons = "Dense"
        else: cons = "Very Dense"
        
        clean.append({
            "borehole": row["borehole"], "depth": depth,
            "N60": N60, "Vs": round(Vs, 1),
            "soil": row["soil"].strip().title(),
            "consistency": cons,
        })
    
    except (ValueError, KeyError) as e:
        error_log.append(f"Row {i}: {e}")

print(f"Step 3: ✓ {len(clean)} clean, {len(error_log)} errors")
for err in error_log:
    print(f"  ⚠ {err}")

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 4: Save clean results + error log
# ══════════════════════════════════════════════════════

# Save clean CSV
if clean:
    fields = ["borehole", "depth", "N60", "Vs", "soil", "consistency"]
    with open("geotech_data/clean_output.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(clean)
    print("Step 4a: ✓ Saved clean_output.csv")

# Save error log
with open("geotech_data/error_log.txt", "w") as f:
    f.write("ERROR LOG\n")
    f.write("=" * 40 + "\n")
    f.write(f"Input: {len(raw_data)} rows\n")
    f.write(f"Clean: {len(clean)} rows\n")
    f.write(f"Errors: {len(error_log)}\n\n")
    for err in error_log:
        f.write(f"  {err}\n")

print("Step 4b: ✓ Saved error_log.txt")
print()

# Show clean results
print("Clean results:")
with open("geotech_data/clean_output.csv", "r") as f:
    print(f.read())

print("Error log:")
with open("geotech_data/error_log.txt", "r") as f:
    print(f.read())

print(f"Pipeline complete: 8 rows in → {len(clean)} clean + {len(error_log)} logged errors")

---

## Practice Exercises

### Exercise 1: Safe float converter
Write `safe_float(value, default=0.0)` that converts any value to float. Test with: `"3.5"`, `"NR"`, `""`, `"6.0m"`

### Exercise 2: Data validator
Write `validate_layer(depth, N60, soil)` that raises `ValueError` if: depth < 0, N60 < 0, N60 > 100, or soil is empty.

### Exercise 3: Safe CSV reader
Write a function that reads a CSV, processes rows inside try/except, collects good rows and errors, and prints a summary.

In [ ]:
# EXERCISE 1
# def safe_float(value, default=0.0):
#     try:
#         return float(value)
#     except ValueError:
#         return default
# 
# Test: safe_float("3.5"), safe_float("NR"), safe_float("6.0m")

In [ ]:
# EXERCISE 2
# def validate_layer(depth, N60, soil):
#     ...

In [ ]:
# EXERCISE 3
# def safe_csv_reader(filepath):
#     ...

---

## Complete Error Handling Reference

### Structure
```python
try:
    risky_code()           # attempt
except ValueError:
    handle_bad_data()      # if conversion fails
except ZeroDivisionError:
    handle_zero()          # if divide by zero
except FileNotFoundError:
    handle_missing()       # if file not found
else:
    on_success()           # only if no error
finally:
    cleanup()              # ALWAYS runs
```

### Common Error Types

| Error | Cause | Fix |
|-------|-------|-----|
| `ValueError` | `int("NR")` | Use default or skip |
| `ZeroDivisionError` | `x / 0` | Check before dividing |
| `FileNotFoundError` | Missing file | Check `os.path.exists()` |
| `KeyError` | Wrong dict key | Use `.get(key, default)` |
| `IndexError` | Bad list index | Check `len()` first |
| `TypeError` | Wrong data type | Convert first |

---

## Key Takeaways

1. **`try/except` = safety net.** Put risky operations (file reading, type conversion, division) inside `try`. The program survives errors instead of crashing.

2. **Always catch SPECIFIC errors** — `except ValueError:`, not bare `except:`. Different errors need different fixes.

3. **"Skip bad, keep good"** is the #1 real-world pattern: wrap each row's processing in `try/except` inside a loop. One bad row doesn't kill the other 999.

4. **`raise`** lets you create engineering-specific validation: reject N60 > 100, negative depths, or impossible values that Python wouldn't catch on its own.

5. **`finally`** always runs — use it to save logs, close connections, or write partial results even when processing fails halfway.

---

## How This Connects to ML

| Today | ML Equivalent |
|-------|---------------|
| `try/except` in data loading | Handling corrupt images, bad CSV rows |
| `raise ValueError` | Input validation in `model.fit()` |
| Skip bad rows | `errors='coerce'` in `pd.to_numeric()` |
| Error logging | TensorBoard, MLflow experiment tracking |
| `finally` cleanup | Releasing GPU memory, saving checkpoints |
| Validate before compute | Schema validation in data pipelines |

**Every ML pipeline needs error handling.** Corrupt images, missing labels, NaN values — models crash on bad data just like our SPT scripts. Today's patterns are the foundation.

---

*Day 009 of 100 — Our code is now crash-proof!*